In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

In [6]:
df = pd.read_parquet('features.parquet')

In [7]:
df_hmm = df.copy()
df_hmm["datetime"] = pd.to_datetime(df_hmm["datetime"], utc=True)
df_hmm = df_hmm.sort_values("datetime").set_index("datetime")

r = df_hmm["r"]
close = df_hmm["close"]
log_close = np.log(close)

df_hmm["ret_6"] = r.rolling(6).sum()
df_hmm["ret_24"] = r.rolling(24).sum()

df_hmm["rv_42"] = np.sqrt((r ** 2).rolling(42).sum())
df_hmm["rv_72"] = np.sqrt((r ** 2).rolling(72).sum())

df_hmm["down_rv_72"] = np.sqrt((np.minimum(r, 0.0) ** 2).rolling(72).sum())
df_hmm["up_rv_72"] = np.sqrt((np.maximum(r, 0.0) ** 2).rolling(72).sum())

df_hmm["downside_share_72"] = df_hmm["down_rv_72"] / (
    df_hmm["down_rv_72"] + df_hmm["up_rv_72"]
)

df_hmm["rv_ratio_24_72"] = (
    np.sqrt((r ** 2).rolling(24).sum()) / df_hmm["rv_72"]
)

df_hmm["ma_dist_72"] = log_close - log_close.rolling(72).mean()
df_hmm["ma_slope_42"] = log_close.rolling(42).mean().diff(6)
df_hmm["drawdown_180"] = close / close.rolling(180).max() - 1

df_hmm["ret_24_over_rv_72"] = df_hmm["ret_24"] / df_hmm["rv_72"]

feature_cols = [
    "r",
    "roll_skew",
    "ret_6",
    "ret_24",
    "rv_42",
    "rv_72",
    "rv_ratio_24_72",
    "downside_share_72",
    "ma_dist_72",
    "ma_slope_42",
    "drawdown_180",
    "ret_24_over_rv_72",
]

df_hmm = (
    df_hmm[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

X = df_hmm[feature_cols].to_numpy(dtype=float)
Y = np.zeros(len(df_hmm), dtype=int)
dates = df_hmm.index

r_col = feature_cols.index("r")

In [9]:
def walk_forward(X, Y, dates, model, tr, v, te, em, **model_kwargs):
    preds = []
    n = len(Y)
    t = np.arange(n)

    for k in range(0, n - tr - v - 2 * em - te + 1, te):
        train = t[k : k + tr]
        val = t[k + tr + em : k + tr + em + v]
        test = t[k + tr + 2 * em + v : k + tr + 2 * em + v + te]

        X_train, Y_train = X[train], Y[train]
        X_val, Y_val = X[val], Y[val]
        X_test = X[test]
        Y_test = Y[test]

        Y_hat = model(
            X_train=X_train,
            Y_train=Y_train,
            X_val=X_val,
            Y_val=Y_val,
            X_test=X_test,
            dates=dates,
            X_all=X,
            train_idx=train,
            val_idx=val,
            test_idx=test,
            **model_kwargs,
        )

        fold = pd.DataFrame(
            {
                "Y_true": Y_test,
                "Y_pred": Y_hat,
            },
            index=dates[test],
        )

        preds.append(fold)

    return pd.concat(preds)

In [10]:
def make_future_return_target(X_all, end_idx, horizon=24, r_col=0):
    y = []

    for i in end_idx:
        future_r = X_all[i + 1 : i + 1 + horizon, r_col]
        y.append(np.sum(future_r))

    return np.array(y, dtype=float)

In [11]:
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1):
        super().__init__()
        pad = dilation * (kernel_size - 1)
        self.net = nn.Sequential(
            nn.ConstantPad1d((pad, 0), 0.0),
            nn.Conv1d(
                in_channels=in_ch,
                out_channels=out_ch,
                kernel_size=kernel_size,
                dilation=dilation,),)
    def forward(self, x):
        return self.net(x)
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            CausalConv1d(in_ch, out_ch, kernel_size, dilation),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Dropout(dropout),
            CausalConv1d(out_ch, out_ch, kernel_size, dilation),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Dropout(dropout),)
        self.res = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        return self.conv(x) + self.res(x)
class TCNAutoencoder(nn.Module):
    def __init__(
        self,
        n_features,
        seq_len,
        latent_dim=4,
        hidden_ch=32,
        kernel_size=3,
        dropout=0.1,):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_ch = hidden_ch
        self.encoder = nn.Sequential(
            TCNBlock(n_features, hidden_ch, kernel_size, dilation=1, dropout=dropout),
            TCNBlock(hidden_ch, hidden_ch, kernel_size, dilation=2, dropout=dropout),
            TCNBlock(hidden_ch, hidden_ch, kernel_size, dilation=4, dropout=dropout),)
        self.to_z = nn.Linear(hidden_ch, latent_dim)
        self.dec_fc = nn.Linear(latent_dim, hidden_ch * seq_len)
        self.decoder = nn.Sequential(
            nn.Conv1d(hidden_ch, hidden_ch, kernel_size=3, padding=1),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Conv1d(hidden_ch, n_features, kernel_size=1),)
    def encode(self, x):
        x = x.transpose(1, 2)          
        h = self.encoder(x)           
        h_last = h[:, :, -1]          
        z = self.to_z(h_last)
        return z
    def forward(self, x):
        z = self.encode(x)
        h = self.dec_fc(z)
        h = h.view(x.shape[0], self.hidden_ch, self.seq_len)
        recon = self.decoder(h)
        recon = recon.transpose(1, 2)  
        return recon, z

In [12]:
def make_causal_windows(X_all_s, end_idx, seq_len):
    windows = []
    for i in end_idx:
        start = i - seq_len + 1
        if start < 0:
            raise ValueError("Not enough history for causal window.")
        windows.append(X_all_s[start : i + 1])
    return np.stack(windows).astype(np.float32)
def fit_tcn_autoencoder(
    X_seq_train,
    n_features,
    seq_len,
    latent_dim=4,
    hidden_ch=32,
    epochs=40,
    batch_size=128,
    lr=1e-3,
    weight_decay=1e-5,
    device=None,):
    if device is None:
        device = "cuda"
    model = TCNAutoencoder(
        n_features=n_features,
        seq_len=seq_len,
        latent_dim=latent_dim,
        hidden_ch=hidden_ch,
    ).to(device)
    ds = TensorDataset(torch.tensor(X_seq_train, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,)
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        for (xb,) in dl:
            xb = xb.to(device)
            recon, _ = model(xb)
            loss = loss_fn(recon, xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model
@torch.no_grad()
def encode_tcn(model, X_seq, device=None):
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    x = torch.tensor(X_seq, dtype=torch.float32).to(device)
    z = model.encode(x)
    return z.cpu().numpy()

In [13]:
def _normalize(p):
    s = p.sum()
    if s <= 0 or not np.isfinite(s):
        return np.ones_like(p) / len(p)
    return p / s
def _filtered_probs_hmm(model, X_seq, init_alpha=None):
    log_lik = model._compute_log_likelihood(X_seq)
    lik = np.exp(log_lik - log_lik.max(axis=1, keepdims=True))
    T, K = lik.shape
    alpha = np.zeros((T, K))
    if init_alpha is None:
        prev = model.startprob_.copy()
    else:
        prev = init_alpha.copy()
    prev = _normalize(prev)
    for t in range(T):
        if t == 0:
            pred = prev
        else:
            pred = alpha[t - 1] @ model.transmat_
        alpha[t] = _normalize(pred * lik[t])
    return alpha

In [14]:
def tcn_hmm_model(
    X_train,
    Y_train,
    dates,
    X_val=None,
    Y_val=None,
    X_test=None,
    **kwargs,
):
    X_all = kwargs["X_all"]
    train_idx = kwargs["train_idx"]
    test_idx = kwargs["test_idx"]

    seq_len = kwargs.get("seq_len", 72)
    latent_dim = kwargs.get("latent_dim", 4)
    hidden_ch = kwargs.get("hidden_ch", 32)
    epochs = kwargs.get("epochs", 40)
    batch_size = kwargs.get("batch_size", 128)
    lr = kwargs.get("lr", 1e-3)

    pred_horizon = kwargs.get("pred_horizon", 24)
    r_col = kwargs.get("r_col", 0)
    n_components = kwargs.get("n_components", 5)

    n_features = X_all.shape[1]

    scaler_x = StandardScaler()
    scaler_x.fit(X_train)

    X_all_s = scaler_x.transform(X_all)

    train_start = train_idx[0]

    train_end_idx = train_idx[
        train_idx >= train_start + seq_len - 1
    ]

    X_seq_train = make_causal_windows(
        X_all_s=X_all_s,
        end_idx=train_end_idx,
        seq_len=seq_len,
    )

    X_seq_test = make_causal_windows(
        X_all_s=X_all_s,
        end_idx=test_idx,
        seq_len=seq_len,
    )

    torch.manual_seed(5)
    np.random.seed(5)

    tcn = fit_tcn_autoencoder(
        X_seq_train=X_seq_train,
        n_features=n_features,
        seq_len=seq_len,
        latent_dim=latent_dim,
        hidden_ch=hidden_ch,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
    )

    Z_train = encode_tcn(tcn, X_seq_train)
    Z_test = encode_tcn(tcn, X_seq_test)

    scaler_z = StandardScaler()
    Z_train_s = scaler_z.fit_transform(Z_train)
    Z_test_s = scaler_z.transform(Z_test)

    hmm = GaussianHMM(
        n_components=n_components,
        covariance_type="full",
        n_iter=1000,
        tol=1e-4,
        random_state=5,
        verbose=False,
        min_covar=1e-6,
        implementation="log",
    )

    hmm.fit(Z_train_s)

    train_filt = _filtered_probs_hmm(hmm, Z_train_s)

    valid_mask = train_end_idx <= train_idx[-1] - pred_horizon
    valid_train_end_idx = train_end_idx[valid_mask]
    valid_train_filt = train_filt[valid_mask]

    y_fwd_ret = make_future_return_target(
        X_all=X_all,
        end_idx=valid_train_end_idx,
        horizon=pred_horizon,
        r_col=r_col,
    )

    regime_score = []

    for k in range(hmm.n_components):
        w = valid_train_filt[:, k]
        score = np.sum(w * y_fwd_ret) / np.sum(w)
        regime_score.append(score)

    regime_score = np.array(regime_score)
    order = np.argsort(regime_score)

    context_start = train_end_idx[-1] + 1
    context_end = test_idx[0]

    init_test = train_filt[-1] @ hmm.transmat_

    if context_start < context_end:
        context_idx = np.arange(context_start, context_end)

        X_seq_context = make_causal_windows(
            X_all_s=X_all_s,
            end_idx=context_idx,
            seq_len=seq_len,
        )

        Z_context = encode_tcn(tcn, X_seq_context)
        Z_context_s = scaler_z.transform(Z_context)

        context_filt = _filtered_probs_hmm(
            hmm,
            Z_context_s,
            init_alpha=init_test,
        )

        init_test = context_filt[-1] @ hmm.transmat_

    test_filt = _filtered_probs_hmm(
        hmm,
        Z_test_s,
        init_alpha=init_test,
    )

    test_filt = test_filt[:, order]

    return [tuple(row) for row in test_filt]

In [16]:
tr = 5000
v = 0
te = 80
em = 4

preds_tcn_hmm = walk_forward(
    X=X,
    Y=Y,
    dates=dates,
    model=tcn_hmm_model,
    tr=tr,
    v=v,
    te=te,
    em=em,
    seq_len=72,
    pred_horizon=24,
    latent_dim=4,
    hidden_ch=32,
    epochs=30,
    batch_size=128,
    lr=1e-3,
    r_col=r_col,
    n_components=5,
)

In [18]:
proba_tcn_hmm = np.vstack(preds_tcn_hmm["Y_pred"].to_numpy())

regime_cols = [f"p_regime_{i}" for i in range(proba_tcn_hmm.shape[1])]

tcn_hmm_probs = pd.DataFrame(
    proba_tcn_hmm,
    index=preds_tcn_hmm.index,
    columns=regime_cols,
)

tcn_hmm_probs.index.name = "datetime"
tcn_hmm_probs = tcn_hmm_probs.reset_index()

tcn_hmm_probs["regime_argmax"] = (
    tcn_hmm_probs[regime_cols]
    .to_numpy()
    .argmax(axis=1)
)

tcn_hmm_probs["regime_confidence"] = (
    tcn_hmm_probs[regime_cols]
    .max(axis=1)
)

tcn_hmm_probs.to_parquet(
    "tcn_hmm_regime_probabilities_oos_filtered.parquet",
    index=False,
    engine="pyarrow",
)

In [15]:
tcn_hmm_probs.head()

,datetime,p_regime_0,p_regime_1,p_regime_2,regime_argmax,regime_confidence
0,2022-07-16 00:00:00+00:00,0.000325,0.596679,0.402997,1,0.596679
1,2022-07-16 04:00:00+00:00,0.000276,0.810089,0.189635,1,0.810089
2,2022-07-16 08:00:00+00:00,0.000269,0.915931,0.083801,1,0.915931
3,2022-07-16 12:00:00+00:00,0.000306,0.948327,0.051367,1,0.948327
4,2022-07-16 16:00:00+00:00,0.000413,0.900740,0.098847,1,0.900740


In [16]:
tcn_hmm_probs[["p_regime_0", "p_regime_1", "p_regime_2"]].sum(axis=1).describe()

count    8.280000e+03
mean     1.000000e+00
std      6.180870e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64